# Lab 4 -- Sensor-Agnostic Posture Classifier (train + convert to TFLite)

Trains a single small neural network that classifies posture (supine/prone/side/sitting/unknown) from a **3-channel (x,y,z) windowed segment from any one of the accelerometer, gyroscope, or magnetometer** -- the same network has to work no matter which sensor fed it.

**How to use this notebook:**
1. Zip the `IMU_Data_Collection/` folder from Lab4 on your own machine (must contain the 9-column CSVs from the updated `IMU_Data_Logger.ino`, i.e. accel+gyro+mag together).
2. Run the cells top to bottom. The upload cell will prompt you to pick that zip file.
3. At the end, download `model_data.h` (the TFLite model as a C array) and note the printed normalization constants -- both go into the Arduino sketch.

## 1. Setup

In [ ]:
import os, io, zipfile, json, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print('TensorFlow version:', tf.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Upload your data
Upload a `.zip` of the `IMU_Data_Collection` folder (the 9-column trial CSVs).

In [ ]:
from google.colab import files

DATA_DIR = Path('IMU_Data_Collection')
DATA_DIR.mkdir(exist_ok=True)

uploaded = files.upload()  # pick your IMU_Data_Collection.zip
for fname in uploaded:
    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[fname])) as zf:
            zf.extractall('.')

# Handle either a zip that contains the folder itself, or one that contains the CSVs directly.
csv_files = [p for p in Path('.').rglob('*.csv') if 'sample_data' not in p.parts]
# ('sample_data' is Colab's own pre-loaded folder, e.g. california_housing_test.csv --
# excluded so it doesn't get swept up and confuse the labeler below.)
csv_files = [p for p in csv_files if p.name != '']
print(f'Found {len(csv_files)} CSV files:')
for p in sorted(csv_files)[:5]:
    print(' ', p)
if len(csv_files) > 5:
    print('  ...')

## 3. Load and label raw trials
Ground-truth label comes from the filename prefix, same scheme as Lab 3: 1=supine, 2=prone, 3=side, 4=sitting, 5=unknown (standing/unknown-* files).

In [ ]:
CLASS_NAMES = {1: 'supine', 2: 'prone', 3: 'side', 4: 'sitting', 5: 'unknown'}
RAW_COLS = ['ax_g', 'ay_g', 'az_g', 'gx_dps', 'gy_dps', 'gz_dps', 'mx_uT', 'my_uT', 'mz_uT']

def label_for_filename(name: str) -> int:
    n = name.lower()
    if n.startswith('supine'): return 1
    if n.startswith('prone'): return 2
    if n.startswith('side'): return 3
    if n.startswith('sitting'): return 4
    if n.startswith('standing') or n.startswith('unknown'): return 5
    raise ValueError(f"Don't know how to label file: {name}")

def load_trial(path):
    df = pd.read_csv(path)
    return df[RAW_COLS].to_numpy(dtype=np.float32)  # shape (n_rows, 9)

files_by_class = defaultdict(list)
for p in csv_files:
    label = label_for_filename(p.name)
    files_by_class[label].append(p)

print('Trials found per class:')
for label in sorted(files_by_class):
    print(f'  {label} ({CLASS_NAMES[label]}): {len(files_by_class[label])} trials')

## 4. Split whole TRIALS into train/val/test
Same reasoning as Lab 3: rows within one trial are highly correlated (same held-still posture), so we split by whole trial file, not by row, to avoid leaking near-duplicate data across splits.

In [ ]:
def group_split(trial_files, seed):
    rng = random.Random(seed)
    trial_files = list(trial_files)
    rng.shuffle(trial_files)
    if len(trial_files) < 3:
        raise ValueError('Need at least 3 trials per class to hold out val+test.')
    test = trial_files[:1]
    val = trial_files[1:2]
    train = trial_files[2:]
    return train, val, test

split_files = {'train': [], 'val': [], 'test': []}
for label, trial_files in files_by_class.items():
    tr, va, te = group_split(trial_files, seed=SEED + label)
    split_files['train'] += [(p, label) for p in tr]
    split_files['val'] += [(p, label) for p in va]
    split_files['test'] += [(p, label) for p in te]

for split in ('train', 'val', 'test'):
    print(f"{split}: {len(split_files[split])} trials ->", [p.name for p, _ in split_files[split]])

## 5. Normalize (per-channel, TRAIN-only stats)
Each of the 9 raw channels gets its own (mean, std), computed from TRAIN rows only and then applied to all splits -- same strategy as Lab 3. This is done on the *raw, pre-window* rows, so the informative DC/mean level of a held-still posture is preserved (per-window normalization would wrongly zero that out). At inference time, the board knows which sensor it's using, so it applies the matching 3 of these 9 constants -- printed at the end of this notebook.

In [ ]:
train_rows = np.concatenate([load_trial(p) for p, _ in split_files['train']], axis=0)
channel_mean = train_rows.mean(axis=0)
channel_std = train_rows.std(axis=0)
channel_std[channel_std == 0] = 1.0

print('Per-channel normalization (from TRAIN only):')
for name, m, s in zip(RAW_COLS, channel_mean, channel_std):
    print(f'  {name:8s} mean={m:9.4f}  std={s:9.4f}')

## 6. Window + sensor-agnostic pooling
For each trial, each of the 3 sensors (accel/gyro/mag) is windowed *independently* into fixed-length, overlapping segments, and all three sensors' windows are pooled into ONE shared dataset under the same class label -- this is what makes the trained model sensor-agnostic: it never sees which sensor a window came from, only 3 generic x/y/z channels.

In [ ]:
WINDOW_SEC = 3.0
SAMPLE_RATE_HZ = 5.0  # matches IMU_Data_Logger.ino's logging rate
WINDOW_LEN = int(WINDOW_SEC * SAMPLE_RATE_HZ)   # samples per window per channel
STRIDE = max(1, WINDOW_LEN // 3)                # overlapping windows -> more real examples per trial

SENSOR_CHANNEL_IDX = {
    'accel': [0, 1, 2],
    'gyro':  [3, 4, 5],
    'mag':   [6, 7, 8],
}

def windows_for_trial(norm_rows, label):
    """norm_rows: (n_rows, 9) already-normalized raw samples for one trial.
    Returns list of (flattened_window (WINDOW_LEN*3,), label, sensor_name)."""
    out = []
    n_rows = norm_rows.shape[0]
    for sensor_name, idx in SENSOR_CHANNEL_IDX.items():
        sensor_data = norm_rows[:, idx]  # (n_rows, 3)
        start = 0
        while start + WINDOW_LEN <= n_rows:
            w = sensor_data[start:start + WINDOW_LEN]  # (WINDOW_LEN, 3)
            out.append((w.flatten(), label, sensor_name))
            start += STRIDE
    return out

def build_split(split_name):
    X, y, sensors = [], [], []
    for path, label in split_files[split_name]:
        raw = load_trial(path)
        norm = (raw - channel_mean) / channel_std
        for flat_window, lbl, sensor_name in windows_for_trial(norm, label):
            X.append(flat_window)
            y.append(lbl - 1)  # 1..5 -> 0..4
            sensors.append(sensor_name)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), np.array(sensors)

X_train, y_train, sensors_train = build_split('train')
X_val, y_val, sensors_val = build_split('val')
X_test, y_test, sensors_test = build_split('test')

print('Window shape (flattened):', X_train.shape[1], f'= {WINDOW_LEN} samples x 3 channels')
for split_name, X, sensors in (('train', X_train, sensors_train), ('val', X_val, sensors_val), ('test', X_test, sensors_test)):
    print(f'{split_name}: {X.shape[0]} windows total', {s: int((sensors == s).sum()) for s in ('accel', 'gyro', 'mag')})

## 7. Build the model

In [ ]:
INPUT_DIM = X_train.shape[1]
N_CLASSES = len(CLASS_NAMES)

model = keras.Sequential([
    keras.layers.Input(shape=(INPUT_DIM,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(N_CLASSES, activation='softmax'),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 8. Train
Early stopping on validation loss -- Lab 3 showed clear overfitting without it, so this time we stop once val_loss stops improving instead of always running the full epoch budget.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=2,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
fig.tight_layout()
fig.savefig('curves.png', dpi=130)
plt.show()

## 9. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}   Test loss: {test_loss:.4f}')

preds = np.argmax(model.predict(X_test, verbose=0), axis=1)
cm = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for t, p in zip(y_test, preds):
    cm[t][p] += 1

names = [CLASS_NAMES[i + 1] for i in range(N_CLASSES)]
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(names)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Test confusion matrix (all sensors pooled)')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.colorbar(im); fig.tight_layout(); fig.savefig('confusion.png', dpi=130); plt.show()

# Also break test accuracy down BY sensor type -- this is the key sensor-agnostic check:
# accuracy should be similar across accel/gyro/mag, not just good on average.
print('\nTest accuracy by sensor type:')
for sensor_name in ('accel', 'gyro', 'mag'):
    mask = sensors_test == sensor_name
    if mask.sum() == 0:
        continue
    acc = (preds[mask] == y_test[mask]).mean()
    print(f'  {sensor_name:6s}: {acc:.4f}  (n={mask.sum()})')

## 10. Convert to TensorFlow Lite
Produces a float32 `.tflite` model. If your TFLite Micro build needs int8, use the quantized cell below instead (needs a representative dataset, provided).

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'Saved model.tflite ({len(tflite_model)} bytes)')

### 10b. (Optional) int8-quantized version -- smaller/faster on-device

In [ ]:
def representative_dataset():
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X_train), size=min(200, len(X_train)), replace=False)
    for i in idx:
        yield [X_train[i:i+1].astype(np.float32)]

converter_q = tf.lite.TFLiteConverter.from_keras_model(model)
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
converter_q.representative_dataset = representative_dataset
tflite_model_quant = converter_q.convert()

with open('model_quant.tflite', 'wb') as f:
    f.write(tflite_model_quant)
print(f'Saved model_quant.tflite ({len(tflite_model_quant)} bytes, vs {len(tflite_model)} float32)')

## 11. Export as a C header for the Arduino sketch
Writes `model_data.h` -- a byte array TensorFlow Lite for Microcontrollers can load directly. Also prints the normalization constants as ready-to-paste C arrays.

In [ ]:
def write_c_header(tflite_bytes, out_path, array_name='g_model'):
    lines = [
        '// Auto-generated by train_sensor_agnostic.ipynb -- do not edit by hand.',
        f'const unsigned int {array_name}_len = {len(tflite_bytes)};',
        f'alignas(8) const unsigned char {array_name}[] = {{',
    ]
    hex_bytes = [f'0x{b:02x}' for b in tflite_bytes]
    for i in range(0, len(hex_bytes), 12):
        lines.append('  ' + ', '.join(hex_bytes[i:i+12]) + ',')
    lines.append('};')
    Path(out_path).write_text('\n'.join(lines))

write_c_header(tflite_model, 'model_data.h')
print('Wrote model_data.h')

print('\n--- Normalization constants (paste into the Arduino sketch) ---')
print('const float CHANNEL_MEAN[9] = {' + ', '.join(f'{m:.6f}f' for m in channel_mean) + '};')
print('const float CHANNEL_STD[9]  = {' + ', '.join(f'{s:.6f}f' for s in channel_std) + '};')
print('// order: ax, ay, az, gx, gy, gz, mx, my, mz')
print(f'// WINDOW_LEN = {WINDOW_LEN} samples per channel (at {SAMPLE_RATE_HZ} Hz, ~{WINDOW_SEC}s)')

## 12. Download the files you need back on your own machine

In [ ]:
from google.colab import files as colab_files
colab_files.download('model_data.h')
colab_files.download('model.tflite')
colab_files.download('curves.png')
colab_files.download('confusion.png')

## Next steps
1. Copy `model_data.h` into the Arduino sketch folder and `#include` it.
2. Hardcode the printed `CHANNEL_MEAN` / `CHANNEL_STD` and `WINDOW_LEN` into the firmware.
3. Use the TensorFlow Lite for Microcontrollers library (provided on the assignment page) to load `g_model` and run inference on a live windowed buffer, normalized the same way as here.
4. Build the base-station app + serial/BLE request protocol (Phase 8).